# Gold Layer: Dimensional Model and Export

This notebook converts the cleaned silver dataset into a star-schema-style gold layer for analytics and reporting. The business purpose is to make posts easier to analyze by user profile, platform, crisis stage, topic, sentiment, emotion, region, language, and time.

## Inputs and Outputs

- Input: the cleaned Silver.csv dataset from the silver pipeline.
- Output: dimension tables and a fact table saved as CSV files in the same folder.
- Transformation: business keys are converted into surrogate keys so the data is easier to join and query downstream.


In [1]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path(".")
SILVER_DATA_PATH = DATA_DIR / "Silver.csv"

# Load the cleaned silver dataset generated by the earlier notebook.
df = pd.read_csv(SILVER_DATA_PATH).copy()


def make_dimension(dataframe, columns, key_name):
    """Create a dimension table with a surrogate key."""
    dimension = dataframe.loc[:, columns].drop_duplicates().reset_index(drop=True)
    dimension[key_name] = range(1, len(dimension) + 1)
    return dimension


dim_users = make_dimension(
    df,
    ["follower_count", "following_count", "account_age_days", "verified_flag", "source_type"],
    "user_id",
)
dim_platform = make_dimension(df, ["platform"], "platform_id")
dim_crisis = make_dimension(df, ["crisis_phase"], "crisis_phase_id")
dim_topic = make_dimension(df, ["topic"], "topic_id")
dim_sentiment = make_dimension(df, ["sentiment_label"], "sentiment_id")
dim_emotion = make_dimension(df, ["emotion"], "emotion_id")
dim_region = make_dimension(df, ["region"], "region_id")
dim_language = make_dimension(df, ["language"], "language_id")
dim_time = make_dimension(df, ["year", "month", "day"], "time_id")

# Preview the generated dimensions before they are attached to the fact table.
dim_users.head()
dim_platform.head()
dim_time.head()

,year,month,day,time_id
0,2025,June,Thursday,1
1,2025,August,Monday,2
2,2025,July,Saturday,3
3,2025,August,Wednesday,4
4,2025,July,Thursday,5


## Build Dimension Tables

The following cells create the core dimensions that describe the content and context of each post. Each dimension is reduced to its distinct business values and assigned a surrogate key so downstream joins stay stable.

In [2]:
dim_users = df[
['follower_count',
'following_count',
'account_age_days',
'verified_flag',
'source_type'
]].drop_duplicates()
dim_users['user_id'] = range(1, len(dim_users)+1)

In [3]:
dim_platform = df[['platform']].drop_duplicates()
dim_platform['platform_id'] = range(1,len(dim_platform)+1)

In [4]:
dim_crisis = df[['crisis_phase']].drop_duplicates()
dim_crisis['crisis_phase_id'] = range(1,len(dim_crisis)+1)

In [5]:
dim_tobic = df[['topic']].drop_duplicates()
dim_tobic ['tobic_id']=range(1,len(dim_tobic)+1)

In [6]:
dim_sentiment = df[['sentiment_label']].drop_duplicates()
dim_sentiment ['sentiment_id']=range(1,len(dim_sentiment)+1)

In [7]:
dim_emotion = df[['emotion']].drop_duplicates()
dim_emotion ['emotion_id']=range(1,len(dim_emotion)+1)

In [8]:
dim_region = df[['region']].drop_duplicates()
dim_region ['region_id']=range(1,len(dim_region)+1)

In [9]:
dim_language = df[['language']].drop_duplicates()
dim_language ['language_id']=range(1,len(dim_language)+1)

In [10]:
dim_time = df[ [ 'year', 'month', 'day' ] ].drop_duplicates().reset_index(drop=True) 
dim_time['time_id'] = range(1, len(dim_time)+1)


## Join Dimensions to the Fact Base

The base post data is enriched with the surrogate keys from each dimension. This creates a gold-level fact table that is easier to query and report on.

In [11]:
df_with_keys = df.copy()

for dimension, source_column, key_column in [
    (dim_platform, "platform", "platform_id"),
    (dim_topic, "topic", "topic_id"),
    (dim_sentiment, "sentiment_label", "sentiment_id"),
    (dim_emotion, "emotion", "emotion_id"),
    (dim_region, "region", "region_id"),
    (dim_language, "language", "language_id"),
    (dim_crisis, "crisis_phase", "crisis_phase_id"),
    (dim_users, ["follower_count", "following_count", "account_age_days", "verified_flag", "source_type"], "user_id"),
]:
    if isinstance(source_column, list):
        # Multi-column merge for dimensions with multiple source columns
        df_with_keys = df_with_keys.merge(
            dimension[[col for col in source_column] + [key_column]], 
            on=source_column, 
            how="left"
        )
    else:
        # Single-column lookup using map
        dimension_lookup = dimension.set_index(source_column)[key_column]
        df_with_keys[key_column] = df_with_keys[source_column].map(dimension_lookup)

# Add time dimension key to the post-level dataset.
df_with_keys = df_with_keys.merge(dim_time.rename(columns={"year": "year", "month": "month", "day": "day"}), on=["year", "month", "day"], how="left")

# Keep the core post-level measures and the surrogate keys for analytics.
fact_posts = [
    "post_id",
    "user_id",
    "platform_id",
    "crisis_phase_id",
    "topic_id",
    "sentiment_id",
    "sentiment_score",
    "emotion_id",
    "stance_label",
    "claim_type",
    "likes",
    "shares",
    "comments",
    "impressions_estimate",
    "reach_estimate",
    "engagement_velocity",
    "misinformation_probability",
    "credibility_score",
    "uncertainty_index",
    "subjectivity_score",
    "toxicity_score",
    "region_id",
    "language_id",
    "is_share",
    "parent_post_id",
    "cascade_depth",
    "time_id",
]

fact_posts_final = df_with_keys.loc[:, fact_posts].copy()

## Create the Fact Table

The fact table keeps the measurable post-level attributes while using the surrogate keys from the dimensions for clean analytical joins.

In [12]:
fact_posts_columns = [
    "post_id",
    "user_id",
    "platform_id",
    "crisis_phase_id",
    "topic_id",
    "sentiment_id",
    "sentiment_score",
    "emotion_id",
    "stance_label",
    "claim_type",
    "likes",
    "shares",
    "comments",
    "impressions_estimate",
    "reach_estimate",
    "engagement_velocity",
    "misinformation_probability",
    "credibility_score",
    "uncertainty_index",
    "subjectivity_score",
    "toxicity_score",
    "region_id",
    "language_id",
    "is_share",
    "parent_post_id",
    "cascade_depth",
    "time_id",
]

fact_posts_final = df_with_keys.loc[:, fact_posts_columns].copy()

In [13]:
output_files = {
    "fact_posts.csv": fact_posts_final,
    "dim_time.csv": dim_time,
    "dim_platform.csv": dim_platform,
    "dim_users.csv": dim_users,
    "dim_sentiment.csv": dim_sentiment,
    "dim_emotion.csv": dim_emotion,
    "dim_region.csv": dim_region,
    "dim_language.csv": dim_language,
    "dim_crisis.csv": dim_crisis,
    "dim_topic.csv": dim_topic,
}

for file_name, dataframe in output_files.items():
    output_path = DATA_DIR / file_name
    dataframe.to_csv(output_path, index=False)
    print(f"Saved {len(dataframe):,} rows to {output_path}")

Saved 50,000 rows to fact_posts.csv
Saved 21 rows to dim_time.csv
Saved 3 rows to dim_platform.csv
Saved 50,000 rows to dim_users.csv
Saved 3 rows to dim_sentiment.csv
Saved 6 rows to dim_emotion.csv
Saved 1 rows to dim_region.csv
Saved 1 rows to dim_language.csv
Saved 5 rows to dim_crisis.csv
Saved 7 rows to dim_topic.csv
